# Financial Research AI Agent

This notebook builds an AI-powered financial research assistant.

The system can:
- Fetch real-time stock prices
- Analyze stock trends using technical indicators
- Plot stock charts
- Analyze financial news sentiment
- Evaluate portfolio performance
- Provide AI-based financial insights

Technologies used:
- LangChain Agents
- Yahoo Finance API (yfinance)
- Python data analysis (pandas)
- Sentiment analysis (TextBlob)
- Financial visualization (matplotlib)
- Groq LLM

In [49]:
import os

# Set your Groq API key here
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

print("API Key loaded successfully")

API Key loaded successfully


## Step 2: Installing Required Packages

We need to install the necessary LangChain packages and other dependencies for our agent.


In [8]:
!{sys.executable} -m pip install yfinance pandas matplotlib textblob duckduckgo-search ddgs gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 2.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 6.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 9.2 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 10.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 861.5/861.5 kB 14.5 MB/s  0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15635 sha256=a0a4650c1c5838906055b7636686b9c9ffd1698051b8fa98c25e8bcdf3ac4e10
  Stored in directory: /Users/mohammedtofiq/Library/Caches/pip/wheels/1e/df/0f/e2bbb22d689b30c681feb5410ab64a2523437b34c8ecfc6476
Successfully built multitasking
  Attempting uninstall: hf-xet
    Found 

In [10]:
%pip install ta textblob


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import sys
!{sys.executable} -m pip install langchain-groq


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import sys
!{sys.executable} -m pip install langchain langchain-groq langchain-community


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
import sys
!{sys.executable} -m pip install langchain==0.1.16 langchain-community==0.0.32

  Using cached langchain-0.1.16-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_community-0.0.32-py3-none-any.whl.metadata (8.5 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_text_splitters-0.0.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... /^C


## Step 3: Importing Core Libraries

Let's import the essential LangChain components and other libraries we'll need.


In [18]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
import ta

In [20]:
import sys
!{sys.executable} -m pip install ddgs


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [21]:
import sqlite3

In [22]:
conn = sqlite3.connect("portfolio.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS portfolio(
    symbol TEXT,
    quantity INTEGER
)
""")

conn.commit()

## Step 4: Setting up the Search Tool

We'll use DuckDuckGo search as one of our tools to allow the agent to search for information on the web.


In [24]:
from duckduckgo_search import DDGS

def search_news(query):
    results = []
    
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            results.append(r["title"])
    
    return " ".join(results)

## Step 5: Creating a Custom Stock Market Tool

Here we define a custom tool that can fetch weather data for any city using the WeatherStack API. This tool will be available to our agent.


In [26]:

def get_stock_price(symbol: str) -> str:
    """
    Get the latest stock price for a company.
    Example input: RELIANCE.NS or TCS.NS
    """

    stock = yf.Ticker(symbol)
    data = stock.history(period="1d")

    if data.empty:
        return "Stock data not found."

    price = data["Close"].iloc[-1]

    return f"Latest price of {symbol} is ₹{price:.2f}"

In [27]:

def analyze_stock_trend(symbol: str) -> str:
    """
    Analyze stock trend using a simple moving average.
    Example: RELIANCE.NS
    """

    stock = yf.Ticker(symbol)
    data = stock.history(period="1mo")

    if data.empty:
        return "No stock data available."

    # Calculate 5-day moving average
    data["MA5"] = data["Close"].rolling(5).mean()

    latest_price = data["Close"].iloc[-1]
    moving_avg = data["MA5"].iloc[-1]

    if latest_price > moving_avg:
        trend = "Uptrend 📈"
    else:
        trend = "Downtrend 📉"

    return f"{symbol} current trend: {trend}. Latest price: ₹{latest_price:.2f}"

In [28]:

def plot_stock_chart(symbol: str) -> str:
    """
    Plot a stock price chart for the last 3 months.
    Example: TCS.NS
    """

    stock = yf.Ticker(symbol)
    data = stock.history(period="3mo")

    if data.empty:
        return "No stock data available."

    plt.figure(figsize=(10,5))
    plt.plot(data.index, data["Close"])
    plt.title(f"{symbol} Stock Price - Last 3 Months")
    plt.xlabel("Date")
    plt.ylabel("Price (INR)")
    plt.grid(True)

    plt.show()

    return f"Displayed stock chart for {symbol}"

In [29]:
def analyze_news_sentiment(query: str):

    news_results = search_news(query)

    sentiment_score = TextBlob(news_results).sentiment.polarity

    if sentiment_score > 0:
        sentiment = "Positive 📈"
    elif sentiment_score < 0:
        sentiment = "Negative 📉"
    else:
        sentiment = "Neutral"

    return f"News sentiment for '{query}' appears {sentiment}"

In [30]:

def analyze_portfolio(symbols: str) -> str:
    """
    Analyze portfolio performance.
    Example input: "TCS.NS, RELIANCE.NS, INFY.NS"
    """

    symbols_list = symbols.split(",")

    results = []

    for sym in symbols_list:
        sym = sym.strip()

        stock = yf.Ticker(sym)
        data = stock.history(period="1mo")

        if data.empty:
            results.append(f"{sym}: No data found")
            continue

        start_price = data["Close"].iloc[0]
        end_price = data["Close"].iloc[-1]

        change = ((end_price - start_price) / start_price) * 100

        results.append(f"{sym}: {change:.2f}% change in last month")

    return "\n".join(results)

In [31]:
def technical_analysis(symbol):

    data = yf.download(symbol, period="3mo")

    data['MA20'] = data['Close'].rolling(window=20).mean()
    data['MA50'] = data['Close'].rolling(window=50).mean()

    rsi = ta.momentum.RSIIndicator(data['Close'])
    data['RSI'] = rsi.rsi()

    latest = data.iloc[-1]

    return f"""
Technical Analysis for {symbol}

Latest Price: ₹{latest['Close']:.2f}

MA20: {latest['MA20']:.2f}
MA50: {latest['MA50']:.2f}

RSI: {latest['RSI']:.2f}

RSI > 70 → Overbought
RSI < 30 → Oversold
"""

In [32]:
def compare_stocks(symbol1, symbol2):

    s1 = yf.download(symbol1, period="1mo")
    s2 = yf.download(symbol2, period="1mo")

    p1 = s1["Close"].iloc[-1]
    p2 = s2["Close"].iloc[-1]

    change1 = ((p1 - s1["Close"].iloc[0]) / s1["Close"].iloc[0]) * 100
    change2 = ((p2 - s2["Close"].iloc[0]) / s2["Close"].iloc[0]) * 100

    return f"""
Stock Comparison

{symbol1}
Price: ₹{p1:.2f}
1 Month Change: {change1:.2f}%

{symbol2}
Price: ₹{p2:.2f}
1 Month Change: {change2:.2f}%

Better performer this month: {symbol1 if change1 > change2 else symbol2}
"""

In [33]:
def moving_average_signal(symbol):

    data = yf.download(symbol, period="1y")

    data["MA50"] = data["Close"].rolling(window=50).mean()
    data["MA200"] = data["Close"].rolling(window=200).mean()

    ma50 = data["MA50"].iloc[-1]
    ma200 = data["MA200"].iloc[-1]

    if ma50 > ma200:
        signal = "BUY signal 📈 (Golden Cross)"
    elif ma50 < ma200:
        signal = "SELL signal 📉 (Death Cross)"
    else:
        signal = "HOLD"

    return f"""
Technical Signal for {symbol}

50 Day Moving Average: {ma50:.2f}
200 Day Moving Average: {ma200:.2f}

Trading Signal: {signal}
"""

In [34]:
def research_report(symbol):

    price = get_stock_price(symbol)
    trend = analyze_stock_trend(symbol)
    tech = technical_analysis(symbol)
    signal = moving_average_signal(symbol)
    sentiment = analyze_news_sentiment(symbol)

    return f"""
📊 STOCK RESEARCH REPORT

Symbol: {symbol}

--------------------------------

PRICE
{price}

--------------------------------

TREND ANALYSIS
{trend}

--------------------------------

TECHNICAL INDICATORS
{tech}

--------------------------------

TRADING SIGNAL
{signal}

--------------------------------

NEWS SENTIMENT
{sentiment}

--------------------------------

AI Summary:
Based on current technical indicators and sentiment, review the trend and signal before making investment decisions.
"""

In [35]:
def add_to_portfolio(symbol, quantity):

    conn = sqlite3.connect("portfolio.db")
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO portfolio VALUES (?,?)",
        (symbol, quantity)
    )

    conn.commit()
    conn.close()

    return f"{quantity} shares of {symbol} added to portfolio."

In [36]:
def view_portfolio():

    conn = sqlite3.connect("portfolio.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM portfolio")

    rows = cursor.fetchall()

    conn.close()

    if not rows:
        return "Portfolio is empty."

    result = "📊 Your Portfolio\n\n"

    for symbol, quantity in rows:
        result += f"{symbol} : {quantity} shares\n"

    return result

In [37]:
def portfolio_value():

    conn = sqlite3.connect("portfolio.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM portfolio")

    rows = cursor.fetchall()

    conn.close()

    if not rows:
        return "Portfolio is empty."

    total = 0
    report = "📊 Portfolio Value\n\n"

    for symbol, quantity in rows:

        data = yf.download(symbol, period="1d")
        price = data["Close"].iloc[-1]

        value = price * quantity
        total += value

        report += f"{symbol} : {quantity} shares | ₹{value:.2f}\n"

    report += f"\nTotal Portfolio Value: ₹{total:.2f}"

    return report

## Step 6: Initializing the Language Model

We create an instance of the ChatOpenAI model that will power our agent's reasoning capabilities.


In [38]:
# llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    # Enable tool calling for this model
    model_kwargs={"tool_choice": "auto"},
    api_key=GROQ_API_KEY
)

In [39]:
ticker_map = {
    "reliance": "RELIANCE.NS",
    "tcs": "TCS.NS",
    "infosys": "INFY.NS",
    "hdfc": "HDFCBANK.NS",
    "wipro": "WIPRO.NS"
}

In [40]:
def detect_symbol(query):

    for name in ticker_map:
        if name in query:
            return ticker_map[name]

    # Detect ticker symbols like TCS.NS
    words = query.split()

    for word in words:
        if ".ns" in word.lower():
            return word.upper()

    return None

In [41]:
def detect_quantity(query):

    words = query.split()

    for word in words:
        if word.isdigit():
            return int(word)

    return None

## Step 9: Creating the Agent

Now we create our agent by combining the language model, our tools (search and weather), and the ReAct prompt template.


In [42]:
def financial_agent(query: str):

    query = query.lower()
    symbol = detect_symbol(query)

    if "research" in query or "report" in query:
        if symbol:
            return research_report(symbol)

    if "price" in query:
        return get_stock_price(symbol)

    if "trend" in query:
        return analyze_stock_trend(symbol)
    
    if "technical" in query and symbol:
        return technical_analysis(symbol)
    
    if "add" in query and "portfolio" in query:

        symbol = detect_symbol(query)
        quantity = detect_quantity(query)

        if symbol and quantity:
            return add_to_portfolio(symbol, quantity)

        return "Please specify stock and quantity."
    
    if "show portfolio" in query or "view portfolio" in query:
        return view_portfolio()

    if "portfolio value" in query:
        return portfolio_value()
    
    if "signal" in query or "buy" in query:
        return moving_average_signal(symbol)

    if "chart" in query:
        return plot_stock_chart(symbol)

    if "news" in query:
        return analyze_news_sentiment(query)

    if "portfolio" in query:
        return analyze_portfolio(symbol)

    if "compare" in query or "vs" in query:
        companies = []

        for name in ticker_map:
            if name in query:
                companies.append(ticker_map[name])

        if len(companies) >= 2:
            return compare_stocks(companies[0], companies[1])

    return "I couldn't understand the request."

## Step 10: Testing the Agent

Let's test our agent with a complex query that requires both searching for information and getting weather data. The agent will need to:
1. Find the capital of Madhya Pradesh
2. Get the current weather for that city


In [43]:
response = financial_agent("price of RELIANCE.NS")
print(response)

Latest price of RELIANCE.NS is ₹1404.80


# Final Step: Deployment

We'll use gradio to deploy our AI Agents. To deploy we'll follow the following steps.

1. Add all our logic for AI Agent into a single function
2. Create an interface using gradio's classes
3. Launch the app

In [44]:
%pip install gradio


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [45]:
import gradio as gr

In [46]:
def get_response(query):
    return financial_agent(query)

In [47]:
iface = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(
        label="Ask a question to your AI Agent",
        placeholder="Ask about stock price, trend, or portfolio...",
        lines=2
    ),
    outputs=gr.Textbox(label="Response", lines=10),
    title="AI Agent with Web Access",
    description="This AI Agent has access to the internet. You can ask anything and it will search the web to get you your answer.",
    examples=[
        ["Differentiate between VectorDB and Vector Store"],
        ["What is RAG model?"],
        ["What is the current market cap of NVIDIA"]
    ]
)

In [48]:
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://cfe9a8a16119f78690.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gradio/blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_

## Summary

This notebook demonstrated how to:
- Set up LangChain agents with custom tools
- Use the ReAct framework for reasoning and acting
- Combine multiple tools (search and weather API) in a single agent
- Execute complex multi-step queries that require both information retrieval and data processing
- Deploy an AI Agent using gradio

The agent successfully found that Bhopal is the capital of Madhya Pradesh and retrieved its current weather conditions by using both the search tool and the weather API tool in sequence.
